In [1]:
!pip install torch --force-reinstall --index-url https://download.pytorch.org/whl/cu118
!pip install torch_geometric pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.7.1+cu118.html

Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 67.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 32.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 91.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 10.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 15.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 14.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 8.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 MB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 905.2/9

In [2]:
import argparse
import json
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric
import pandas as pd
import numpy as np

from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_add_pool
from sklearn import metrics

In [3]:
class GINLapPE(nn.Module):
    """
    GIN with Laplacian Positional Encoding (LapPE).

    LapPE is injected at the input by summing a learned projection of the
    Laplacian eigenvectors with the encoded node features before the GIN
    layers, following the GPS paper convention but without global attention.
    """

    def __init__(self, input_dim, hidden_dim, num_layers, dropout, pe_dim):
        super().__init__()
        self.dropout = dropout

        # Projects raw node features into hidden_dim
        self.input_encoder = nn.Linear(input_dim, hidden_dim)
        # Projects pe_dim Laplacian eigenvectors into hidden_dim
        self.pe_encoder = nn.Linear(pe_dim, hidden_dim)

        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        for _ in range(num_layers):
            # All layers operate in hidden_dim (PE injection already happened)
            mlp = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.convs.append(GINConv(mlp))
            self.batch_norms.append(nn.BatchNorm1d(hidden_dim))

        # Graph-level readout sums over all layers (Xu et al. Sec 4.2)
        self.lin1 = nn.Linear(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        pe = data.laplacian_eigenvector_pe

        x = self.input_encoder(x) + self.pe_encoder(pe)

        h_graph = 0
        for conv, batch_norm in zip(self.convs, self.batch_norms):
            x = F.relu(batch_norm(conv(x, edge_index)))
            x = F.dropout(x, self.dropout, training=self.training)
            h_graph = h_graph + global_add_pool(x, batch)

        h = F.relu(self.lin1(h_graph))
        h = F.dropout(h, self.dropout, training=self.training)
        return self.classifier(h).view(-1)

In [4]:
def train(model, optimizer, criterion, train_loader, device):
    model.train()
    total_loss = 0

    for data in train_loader:
        data = data.to(device)
        out = model(data)
        loss = criterion(out, data.y.float())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * data.num_graphs

    return total_loss / len(train_loader.dataset)


@torch.no_grad()
def test(model, loader, device):
    model.eval()
    predictions = []
    labels = []

    for data in loader:
        data = data.to(device)
        out = model(data)
        pred = (out > 0).float()
        predictions.append(pred.cpu())
        labels.append(data.y.cpu())

    accuracy = metrics.accuracy_score(torch.cat(labels), torch.cat(predictions))
    f1 = metrics.f1_score(torch.cat(labels), torch.cat(predictions))

    return accuracy, f1

In [5]:
parser = argparse.ArgumentParser(
    description="GIN+LapPE for partial automorphism extension problem"
)
parser.add_argument(
    "--seed", type=int, default=43, help="Random seed for reproducibility (default: 43)"
)
parser.add_argument(
    "--config",
    type=str,
    default="/kaggle/input/datasets/samuelvarchol/bachelor-thesis-dataset/best_config.json",
    help="Use config file for hyperparameters",
)
parser.add_argument("--hidden_dim", type=int, help="Hidden dimension size")
parser.add_argument("--num_layers", type=int, help="Number of GIN layers")
parser.add_argument("--dropout", type=float, help="Dropout rate")
parser.add_argument("--lr", type=float, help="Learning rate")
parser.add_argument("--weight_decay", type=float, help="Weight decay")
parser.add_argument("--batch_size", type=int, help="Input batch size")
parser.add_argument(
    "--factor",
    type=float,
    default=0.5,
    help="Factor for learning rate scheduler (default: 0.5)",
)
parser.add_argument(
    "--patience",
    type=int,
    default=10,
    help="Patience for learning rate scheduler (default: 10)",
)
parser.add_argument("--epochs", type=int, default=150, help="Number of epochs to train")

args, remaining_argv = parser.parse_known_args()

with open(args.config, "r") as f:
    config_dict = json.load(f)
parser.set_defaults(**config_dict)

args = parser.parse_args("")

In [6]:
PE_DIM = 5
train_dataset = torch.load(
    "/kaggle/input/datasets/samuelvarchol/bachelor-thesis-dataset/train_dataset_with_pe.pt",
    weights_only=False,
)
val_dataset = torch.load(
    "/kaggle/input/datasets/samuelvarchol/bachelor-thesis-dataset/val_dataset_with_pe.pt",
    weights_only=False,
)

number_of_features = train_dataset[0].num_node_features

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [7]:
# ── Seeds used across runs (standard practice in GNN papers) ──────────────────
SEEDS = [0, 1, 2, 3, 4]


def run_single_seed(seed):
    """Full training pipeline for a single seed. Returns best val_acc, val_f1, and history."""
    torch_geometric.seed_everything(seed)

    train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=args.batch_size)

    model = GINLapPE(
        number_of_features,
        args.hidden_dim,
        args.num_layers,
        args.dropout,
        PE_DIM,
    ).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=args.lr, weight_decay=args.weight_decay
    )
    criterion = nn.BCEWithLogitsLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=args.factor, patience=args.patience
    )

    best_val_acc, best_val_f1 = 0.0, 0.0
    patience_counter = 0
    training_history = []
    best_state_dict = None

    for epoch in range(1, args.epochs + 1):
        train_loss = train(model, optimizer, criterion, train_loader, device)
        train_acc, _ = test(model, train_loader, device)
        val_acc, val_f1 = test(model, val_loader, device)
        scheduler.step(val_acc)

        training_history.append(
            {
                "seed": seed,
                "epoch": epoch,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_acc": val_acc,
                "val_f1": val_f1,
                "lr": optimizer.param_groups[0]["lr"],
            }
        )

        if val_acc > best_val_acc:
            best_val_acc, best_val_f1 = val_acc, val_f1
            patience_counter = 0
            best_state_dict = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1

        if patience_counter >= 15:
            print(f"  [seed {seed}] Early stopping at epoch {epoch}.")
            break

        if epoch % 10 == 0:
            print(
                f"  [seed {seed}] Epoch {epoch:03d} | "
                f"loss {train_loss:.4f} | "
                f"train acc {train_acc:.4f} | "
                f"val acc {val_acc:.4f} | val f1 {val_f1:.4f}"
            )

    return best_val_acc, best_val_f1, training_history, best_state_dict


# ── Run all seeds ─────────────────────────────────────────────────────────────
all_val_acc, all_val_f1 = [], []
all_results = []  # (seed, val_acc, val_f1, history, state_dict)

for seed in SEEDS:
    print(f"\n{'=' * 55}\nRun seed={seed}")
    val_acc, val_f1, history, state_dict = run_single_seed(seed)
    all_val_acc.append(val_acc)
    all_val_f1.append(val_f1)
    all_results.append((seed, val_acc, val_f1, history, state_dict))
    print(f"  → best val acc: {val_acc:.4f}  |  best val f1: {val_f1:.4f}")

# ── Aggregate results (paper-style reporting) ─────────────────────────────────
acc_arr = np.array(all_val_acc)
f1_arr = np.array(all_val_f1)

print(f"\n{'=' * 55}")
print("Multi-seed results (mean ± std over 5 runs):")
print(f"  Val Accuracy : {acc_arr.mean():.4f} ± {acc_arr.std():.4f}")
print(f"  Val F1 Score : {f1_arr.mean():.4f}  ± {f1_arr.std():.4f}")
print(f"{'=' * 55}")
print("Per-seed breakdown:")
for seed, acc, f1, _, __ in all_results:
    print(f"  Seed {seed}: acc={acc:.4f}  f1={f1:.4f}")

# ── Save only the global best model's checkpoint and history ──────────────────
best_seed, best_acc, best_f1, best_history, best_state = max(
    all_results, key=lambda x: x[1]
)

torch.save(best_state, "/kaggle/working/best_model.pt")
pd.DataFrame(best_history).to_csv("/kaggle/working/best_model_history.csv", index=False)

print(f"\nSaved checkpoint + history for seed={best_seed} (val_acc={best_acc:.4f})")

# ── Save summary table for all seeds ─────────────────────────────────────────
summary_df = pd.DataFrame(
    {
        "seed": SEEDS,
        "val_acc": all_val_acc,
        "val_f1": all_val_f1,
    }
)
summary_df.loc[len(summary_df)] = ["mean", acc_arr.mean(), f1_arr.mean()]
summary_df.loc[len(summary_df)] = ["std", acc_arr.std(), f1_arr.std()]
summary_df.to_csv("/kaggle/working/multi_seed_summary.csv", index=False)
print("Saved: multi_seed_summary.csv")


Run seed=0
  [seed 0] Epoch 010 | loss 0.4748 | train acc 0.7678 | val acc 0.7449 | val f1 0.7646
  [seed 0] Epoch 020 | loss 0.4242 | train acc 0.7917 | val acc 0.7649 | val f1 0.7941
  [seed 0] Epoch 030 | loss 0.3862 | train acc 0.8189 | val acc 0.7718 | val f1 0.7873
  [seed 0] Epoch 040 | loss 0.3716 | train acc 0.8341 | val acc 0.7821 | val f1 0.7931
  [seed 0] Epoch 050 | loss 0.2822 | train acc 0.8758 | val acc 0.7839 | val f1 0.8038
  [seed 0] Epoch 060 | loss 0.2732 | train acc 0.8959 | val acc 0.7870 | val f1 0.8016
  [seed 0] Epoch 070 | loss 0.2025 | train acc 0.9251 | val acc 0.7845 | val f1 0.7942
  [seed 0] Epoch 080 | loss 0.1965 | train acc 0.9335 | val acc 0.7800 | val f1 0.7815
  [seed 0] Early stopping at epoch 87.
  → best val acc: 0.7922  |  best val f1: 0.8086

Run seed=1
  [seed 1] Epoch 010 | loss 0.4674 | train acc 0.7454 | val acc 0.7245 | val f1 0.7117
  [seed 1] Epoch 020 | loss 0.4168 | train acc 0.7987 | val acc 0.7718 | val f1 0.7968
  [seed 1] Epoch 0